In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import statsmodels.api as sm
from statsmodels.formula.api import ols
from bct import modularity, clustering, centrality
import statsmodels.stats.api as sms

In [4]:
region_label = pd.read_csv('../dataset/Study2/brainmap.csv')
yeo = pd.read_csv('../dataset/Study2/Yeo.csv')

In [5]:
def map_component(x):
    if 1 <= x <= 2:
        return 1
    elif 3 <= x <= 4:
        return 2
    elif 5 <= x <= 6:
        return 3
    elif 7 <= x <= 8:
        return 4
    elif 9 <= x <= 10:
        return 5
    elif 11 <= x <= 13:
        return 6
    elif 14 <= x <= 17:
        return 7
    else:
        return np.nan
region_label['schaefer17_combined'] = [map_component(x) for x in region_label.schaefer17]

In [6]:
names = ['Visual', 'SomMot', 'DorsAttn', 'SalVentAttn', 'Limbic', 'Cont', 'Default']
labels = [f"{row}_{col}" for row in names for col in names]

In [7]:
BCP = glob.glob('../dataset/Study1/schaefer_timeseries/*.csv')
CBPD = glob.glob('../dataset/Study2/correlation/*.tsv')

In [10]:
bcpdemo = pd.read_csv('../dataset/Study1/BCPbasicparas.csv')
with open('../dataset/Study2/CBPD_data_cleaned_DMD_2024.08.24.csv', 'r', encoding='utf-8', errors='ignore') as f:
    cbpddemo = pd.read_csv(f)
cbpdmotion = pd.read_csv('../dataset/Study2/qc.csv')
cbpdmotion = cbpdmotion.loc[:, ['sub', 'ses', 'run', 'fd_mean']]

In [11]:
bcp_all = []
ids = []
for bcp in BCP[:100]:
    df = pd.read_csv(bcp)
    df = np.tanh(df.corr())
    df.columns = region_label['schaefer17_combined'].values
    df.index = region_label['schaefer17_combined'].values
    df_long = df.reset_index().melt(id_vars='index', var_name='column', value_name='value')
    bcp_all.append(df_long.groupby(['index', 'column']).mean()['value'].values)
    ids.append(os.path.basename(bcp).replace('.csv', ''))

In [12]:
bcp_all = pd.DataFrame(bcp_all)
bcp_all.columns = labels
bcp_all['ID'] = ids

In [13]:
region_label = [
   1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
   1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
   2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3,
   3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4,
   4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5,
   5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
   6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7,
   7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
   7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
   7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
   1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
   2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3,
   3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4,
   4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
   5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
   6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7,
   7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
   7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
   7, 7, 7, 7
]
corall = []
ids = []
for bcp in BCP[:100]:
    df = pd.read_csv(bcp)
    corr_matrix = df.corr().values
    z_matrix = np.tanh(corr_matrix)  # Fisher z-transform (optional)
    corr_df = pd.DataFrame(z_matrix)
    corr_df.columns = region_label
    corr_df.index = region_label
    df_long = corr_df.reset_index().melt(id_vars='index', var_name='column', value_name='value')
    avg_network_corr = df_long.groupby(['index', 'column'])['value'].mean().values
    corall.append(avg_network_corr)

In [15]:
# clustering, participation
C_all = []
P_all = []
ids = []
sess = []
for t, cor in enumerate(CBPD):
    if t % 20 == 0:
        print(t / len(CBPD))
    try:
        df = pd.read_csv(cor, sep='\t').iloc[:400, 1:401].fillna(0)
        df.columns = np.arange(400)
        # Identify valid rows/columns
        valid_rows = ~(df == 0).all(axis=1)
        valid_cols = ~(df == 0).all(axis=0)
        df_valid = df.loc[valid_rows, df.columns[valid_cols]]
        if df_valid.shape[0] < 2 or df_valid.shape[0] != df_valid.shape[1]:
            print(f"Skipping (non-square or too small): {cor}")
            C_all.append([None] * 400)
            P_all.append([None] * 400)
            continue
        df_matrix = np.tanh(df_valid.to_numpy())
        try:
            ci, q = modularity.community_louvain(df_matrix, B='negative_asym')
            clustering_vals = np.mean(clustering.clustering_coef_wu_sign(df_matrix), axis=0)
            participation_vals = np.mean(centrality.participation_coef_sign(df_matrix, ci), axis=0)
        except Exception as e:
            print(f"Skipping (error in computation): {cor}, Error: {e}")
            C_all.append([None] * 400)
            P_all.append([None] * 400)
            continue
        # Fill full 400-length vectors
        C_full = [None] * 400
        P_full = [None] * 400
        valid_idx = valid_rows[valid_rows].index.tolist()
        for i, idx in enumerate(valid_idx):
            C_full[idx] = clustering_vals[i]
            P_full[idx] = participation_vals[i]
        C_all.append(C_full)
        P_all.append(P_full)
        ids.append(os.path.basename(cor))
    except:
        continue

0.0


In [16]:
ids = [x.replace('_acq-abcd', '') for x in ids]

In [17]:
cbpddemo['sub'] = [x.split('_')[0] for x in cbpddemo.record_id]

In [ ]:
df_clustering = pd.DataFrame([np.mean(x, axis=0) for x in C_all])
df_participation = pd.DataFrame([np.mean(x, axis=0) for x in P_all])
df_clustering['ID'] = ids
df_participation['ID'] = ids
df_clustering.to_csv('../dataset/Study2/cbpd_clustering.csv', index=False)
df_participation.to_csv('../dataset/Study2/cbpd_participation.csv', index=False)

In [20]:
cbpd_all = []
ids = []
for cbpd in CBPD:
    try:
        df = pd.read_csv(cbpd, sep='\t').iloc[:, 1:]
        df = np.tanh(df)
        if df.dropna().shape[0] == 400:
            df.columns = region_label['schaefer17_combined'].values
            df.index = region_label['schaefer17_combined'].values
            df_long = df.reset_index().melt(id_vars='index', var_name='column', value_name='value')
            cbpd_all.append(df_long.groupby(['index', 'column']).mean()['value'].values)
            ids.append(os.path.basename(cbpd).replace('.tsv', ''))
    except:
        continue

In [21]:
cbpd_all = pd.DataFrame(cbpd_all)
cbpd_all.columns = labels
cbpd_all['ID'] = ids

ValueError: Length mismatch: Expected axis has 0 elements, new values have 49 elements

In [ ]:
bcpdemo['ID'] = bcpdemo.sub_id.astype(str) + '_' + bcpdemo.ses_id.astype(str).str.zfill(2)

In [ ]:
bcp_all = pd.merge(bcp_all, bcpdemo, on='ID')

In [ ]:
cbpd_all['ID'] = [x.replace('acq-abcd_', '') for x in cbpd_all.ID]

In [ ]:
cbpd_all['ses'] = [x.split('_')[1] for x in cbpd_all.ID]
cbpd_all['run'] = [int(x.split('_')[3].replace('run-', '')) for x in cbpd_all.ID]
cbpd_all['ID'] = [x.split('_')[0].replace('sub-', '') for x in cbpd_all.ID]

# Behavior

In [ ]:
#Study1
itsea = pd.read_csv('../dataset/Study1/itsea01.txt', sep='\t')
itsea = itsea.dropna(subset='imitation_play_percentile').loc[:, ['src_subject_id', 'interview_age', 'imitation_play_percentile']].iloc[1:, :]
itsea['src_subject_id'] = itsea['src_subject_id'].astype('float')
itsea['interview_age'] = itsea['interview_age'].astype('float')

In [22]:
#Study2
with open('../dataset/Study2/CBPD_data_cleaned_DMD_2024.08.24.csv', 'r', encoding='utf-8', errors='ignore') as f:
    litnum = pd.read_csv(f)
demo = litnum.loc[:, ['record_id', 'male', 'age_ques','age_scan', 'parent1_edu', 'parent2_edu', 'income_median']]
coding = pd.read_csv('../dataset/Study2/codinglitnum.csv')
litnum = litnum.loc[:, coding.iloc[:, 2].dropna().values].T
coding = coding.loc[:, ['play apm', 'Pretend play']].iloc[:40, :].fillna(0)
coding['Pretend play'] = [1 if x=='X' else 0 for x in coding['Pretend play']]
litnum['play'] = coding['play apm'].values
litnum['pretend'] = coding['Pretend play'].values

In [23]:
demo['litnum_play'] = litnum.groupby('play').mean().reset_index(drop=True).iloc[1, :-1].values
demo['litnum_pretend'] = litnum.groupby('pretend').mean().reset_index(drop=True).iloc[1, :-1].values

# Combine

In [25]:
bcp_all = pd.merge(bcp_all, itsea, left_on=['sub_id'], right_on=['src_subject_id'], how='outer')

In [56]:
bcp_all.scan_age = bcp_all.scan_age / 4
bcp_all['age_gap'] = [np.abs(x) for x in bcp_all.interview_age - bcp_all.scan_age]

In [58]:
bcp_all.to_csv('../dataset/Study1/study1_all_yeo7.csv', index=False)

In [28]:
cbpd_all = cbpd_all_copy.copy()

In [29]:
cbpd_all['ses'] = [int(x.replace('ses-', '')) for x in cbpd_all.ses]
cbpd_all = pd.merge(cbpd_all, cbpdmotion, left_on=['ID', 'ses', 'run'], right_on=['sub', 'ses', 'run'])
cbpd_all = cbpd_all.groupby(['ID', 'ses', 'sub']).mean().reset_index()

In [30]:
cbpd_all['record_id'] = [x+'_'+str(y) if y == 2 else x for x,y in zip(cbpd_all.ID, cbpd_all.ses)]

In [31]:
cbpd_all = pd.merge(cbpd_all, demo, on=['record_id'], how='outer')

In [32]:
cbpd_all['sub_id'] = [x.split('_')[0] for x in cbpd_all.record_id]

In [33]:
cbpd_all.to_csv('../dataset/Study2/study2_all_yeo7.csv', index=False)